Load and profile the raw data

In [11]:
import pandas as pd

cols = ["player_id","age","gender","state","platform","genre",
        "daily_hours","spend_inr","signup_year","esports_participant","churn_risk"]

df = pd.read_csv("../data/raw/indian_gaming_industry_bigdata_50m.csv",
                 header=None, names=cols,
                 dtype={"gender":"category","state":"category",
                        "platform":"category","genre":"category"})

In [4]:
df.shape

(1000000, 11)

In [6]:
df.head()

,player_id,age,gender,state,platform,genre,daily_hours,spend_inr,signup_year,esports_participant,churn_risk
0,IND_GAMER_049000001,40,Male,Delhi NCR,Mobile (Android/iOS),Battle Royale (BGMI/Free Fire),2.8,1898.85,2022,False,False
1,IND_GAMER_049000002,39,Male,Maharashtra,Mobile (Android/iOS),Esports & Tactical Shooters (Valorant/CS),6.0,6903.72,2024,False,False
2,IND_GAMER_049000003,28,Female,Telangana,Console (PS5/Xbox),Action-RPG & Souls-like,4.3,2507.75,2025,True,False
3,IND_GAMER_049000004,22,Female,Delhi NCR,Mobile (Android/iOS),"Casual, Puzzle & Strategy",1.6,1039.96,2024,False,True
4,IND_GAMER_049000005,24,Male,Rajasthan,Mobile (Android/iOS),Cricket & Sports Simulation,5.3,7198.66,2025,False,False


In [7]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 11 columns):
 #   Column               Non-Null Count    Dtype   
---  ------               --------------    -----   
 0   player_id            1000000 non-null  str     
 1   age                  1000000 non-null  int64   
 2   gender               1000000 non-null  category
 3   state                1000000 non-null  category
 4   platform             1000000 non-null  category
 5   genre                1000000 non-null  category
 6   daily_hours          1000000 non-null  float64 
 7   spend_inr            1000000 non-null  float64 
 8   signup_year          1000000 non-null  int64   
 9   esports_participant  1000000 non-null  bool    
 10  churn_risk           1000000 non-null  bool    
dtypes: bool(2), category(4), float64(2), int64(2), str(1)
memory usage: 62.0 MB


In [8]:
df.describe()

,age,daily_hours,spend_inr,signup_year
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,29.502943,2.202525,3750.294251,2023.861029
std,10.390854,2.204563,2165.005179,1.476324
min,12.000000,0.000000,0.000000,2021.000000
25%,21.000000,0.600000,1875.197500,2023.000000
50%,29.000000,1.500000,3754.915000,2024.000000
75%,39.000000,3.100000,5622.410000,2025.000000
max,47.000000,29.300000,7500.000000,2026.000000


In [9]:
df.isna().sum()

player_id              0
age                    0
gender                 0
state                  0
platform               0
genre                  0
daily_hours            0
spend_inr              0
signup_year            0
esports_participant    0
churn_risk             0
dtype: int64

In [10]:
df["player_id"].duplicated().sum()

np.int64(0)

CLEANIND AND VALIDATING THE DATA

In [12]:
import pandas as pd, numpy as np

cols = ["player_id","age","gender","state","platform","genre",
        "daily_hours","spend_inr","signup_year","esports_participant","churn_risk"]
df = pd.read_csv("../data/raw/indian_gaming_industry_bigdata_50m.csv",
                 header=None, names=cols,
                 dtype={"gender":"category","state":"category",
                        "platform":"category","genre":"category"})
print("Rows loaded:", len(df))

Rows loaded: 1000000


In [13]:
print("Missing values:", df.isna().sum().sum()) 

Missing values: 0


In [14]:
print("Duplicate IDs:", df.player_id.duplicated().sum())

Duplicate IDs: 0


In [15]:
print("Hours > 24:", (df.daily_hours > 24).sum()) 

Hours > 24: 16


In [16]:
print("Hours > 12:", (df.daily_hours > 12).sum()) 

Hours > 12: 4217


In [17]:
print("Zero hours:", (df.daily_hours == 0).sum())

Zero hours: 22406


In [18]:
print("Under 18:", (df.age < 18).sum())

Under 18: 166654


In [19]:
print("Spend range:", df.spend_inr.min(), df.spend_inr.max())

Spend range: 0.0 7500.0


In [20]:
#Removing impossible values: more than 24 hours in a day
df = df[df.daily_hours <= 24].copy()

In [21]:
# Flaging, questionable records
df["extreme_hours_flag"] = df.daily_hours > 12
df["zero_hours_flag"]    = df.daily_hours == 0

In [22]:
print("Rows after cleaning:", len(df))   # expect 999,984

Rows after cleaning: 999984


FEATURE ENGINEERING

In [24]:
#Grouping age
df["age_group"] = pd.cut(df.age, bins=[0,17,24,34,47],
                         labels=["Under 18","18-24","25-34","35-47"])

In [25]:
#Adding spending tier
df["spend_tier"] = pd.cut(df.spend_inr, bins=[-1,1500,3750,5625,7500],
                          labels=["Low","Mid","High","Very High"])

In [26]:
# Spend per hour, only where hours > 0
df["spend_per_hour"] = np.where(df.daily_hours > 0,
                                df.spend_inr / df.daily_hours, np.nan)

In [28]:
#State map
region = {"Delhi NCR":"North","Uttar Pradesh":"North","Rajasthan":"North",
          "Maharashtra":"West","Gujarat":"West",
          "Karnataka":"South","Telangana":"South","Tamil Nadu":"South",
          "Bihar":"East","West Bengal":"East","Madhya Pradesh":"Central"}
df["region"] = df.state.map(region)

In [29]:
df.region.isna().sum()

np.int64(0)

In [30]:
#Saving cleaned data
df.to_csv("../data/clean/gaming_clean.csv", index=False)